# 04 — The five report cases + type × level heatmaps (B-25)

Requested by @hwupu 2026-11-02 (P-009, D-25). One case each, **top-1 by a quantitative score** (spec §7: no cherry-picking):

| case | question | score (per image or pair) |
|---|---|---|
| A-1 | SSIM vs MS-SSIM disagree | $\lvert r_{SSIM} - r_{MS\text{-}SSIM}\rvert$ (global percentile ranks) |
| A-2 | LPIPS-VGG vs DISTS disagree | $\lvert r_{LPIPS} - r_{DISTS}\rvert$ |
| B-1 | a metric vs MOS, **rank** view (what SRCC sees) | $\lvert r_m - r_{MOS}\rvert$ over all 5 metrics |
| B-2 | a metric vs MOS, **value** view (what PLCC sees) | $\lvert f_m(x) - MOS\rvert$, $f_m$ = the VQEG logistic used for PLCC |
| C-1 | SSIM vs LPIPS-VGG **ranking** of two images of one reference | $\min(\lvert\Delta r_{SSIM}\rvert, \lvert\Delta r_{LPIPS}\rvert)$, opposite signs, within-reference ranks; human ties ($\lvert\Delta MOS\rvert<0.5$) skipped |

Same filters as notebook 03 (D-24): identical pairs dropped, `mos_var` > 90th percentile not selectable.
Each case takes the first candidate whose image(s) are not already used by an earlier case, so the five cases show five different images.

In [ ]:
import matplotlib.pyplot as plt
import pandas as pd
from humper import failures as F
from humper.data import REPO_ROOT
from humper.evaluate import fit_logistic, srcc
from humper.metrics import METRICS, aligned
from humper.viz import LABELS, case_heatmaps, explain_figure, explain_pair_figure, msssim_scales_figure, srcc_grid_figure

FIG = REPO_ROOT / "results" / "figures" / "selected"; FIG.mkdir(parents=True, exist_ok=True)
OUT = REPO_ROOT / "results" / "failures"
s = pd.read_csv(REPO_ROOT / "results" / "scores.csv")
E = F.eval_set(s)                        # 9,805 non-identical pairs
rg, rr = F.ranks(E), F.ranks(E, by="ref_id")  # global / within-reference percentile ranks
byid = E.set_index("id")
rg.index = rr.index = E["id"]

In [ ]:
A = F.metric_pairs(s)
B1 = F.vs_human(s)
B2 = F.logistic_residuals(s)
C = F.family_pairs(s, trad=["ssim"], learned=["lpips_vgg"])
C = C[C["human_sides_with"] != "tie"]

used = set()
def first_new(df, cols):
    for pos, (_, row) in enumerate(df.iterrows()):
        if not {row[c] for c in cols} & used:
            used.update(row[c] for c in cols)
            return pd.concat([row, pd.Series({"candidate_pos": pos + 1})])

picks = {
    "A-1": first_new(A[A["pair"] == "ssim~ms_ssim"], ["id"]),
    "A-2": first_new(A[A["pair"] == "lpips_vgg~dists"], ["id"]),
    "B-1": first_new(B1, ["id"]),
    "B-2": first_new(B2, ["id"]),
    "C-1": first_new(C, ["id_i", "id_j"]),
}
for k, p in picks.items():
    print(k, "candidate #%d" % p["candidate_pos"], dict(p.drop(["candidate_pos"]).items()), "\n")
print("B-2 top-1 overall:", dict(B2.iloc[0][["id", "metric", "residual"]].items()))
pd.DataFrame(picks).T.to_csv(OUT / "selected_report5.csv")

## Scores and ranks of the selected images (all metrics; r = percentile rank, higher = better)

In [ ]:
ids = [picks[k]["id"] for k in ["A-1", "A-2", "B-1", "B-2"]] + [picks["C-1"]["id_i"], picks["C-1"]["id_j"]]
names = [m.name for m in METRICS]
tbl = byid.loc[ids, ["dist_name", "level", "mos", "mos_var"] + names].join(rg.loc[ids, names + ["mos"]].add_prefix("r_").round(3))
tbl.join(rr.loc[ids, names + ["mos"]].add_prefix("rref_").round(3)).to_csv(OUT / "selected_report5_scores.csv")
tbl

## Heatmaps: distortion type × level

For each case, the **signed** version of its score, averaged per cell over all 81 references (left) and for the selected image's reference only (right; one image per cell). The black box marks the selected image(s).
Red = the first-named quantity is higher (e.g. SSIM ranks the image better than MS-SSIM does, or the metric is more optimistic than humans).

In [ ]:
def fitted_mos(metric):
    f = pd.Series(fit_logistic(aligned(s, metric), s["mos"]), index=s.index)
    return (f[s["psnr"] < 80].to_numpy() - E["mos"].to_numpy())

m1, m2 = picks["B-1"]["metric"], picks["B-2"]["metric"]
measures = {
    "A-1": (rg["ssim"] - rg["ms_ssim"], "r_SSIM − r_MS-SSIM"),
    "A-2": (rg["lpips_vgg"] - rg["dists"], "r_LPIPS − r_DISTS"),
    "B-1": (rg[m1] - rg["mos"], f"r_{LABELS[m1]} − r_MOS"),
    "B-2": (pd.Series(fitted_mos(m2)), f"f_{LABELS[m2]}(x) − MOS  [MOS units]"),
    "C-1": (rr["ssim"] - rr["lpips_vgg"], "within-ref r_SSIM − r_LPIPS"),
}
grids = {}
for k, (vals, label) in measures.items():
    p = picks[k]
    rows = [(p["dist_i"], p["level_i"]), (p["dist_j"], p["level_j"])] if k == "C-1" else [(p["dist_name"], p["level"])]
    g_all, g_ref = F.grid(E, vals), F.grid(E, vals, ref_id=int(p["ref_id"]))
    grids[k] = g_all
    title = f"{k}: {label}" + ("" if k == "C-1" else f" — selected {p['id']}")
    fig = case_heatmaps(g_all, g_ref, int(p["ref_id"]), rows, label, title)
    fig.savefig(FIG / f"{k}_heatmap.png", dpi=130, bbox_inches="tight")
    plt.show()

Per-cell means for the report (top cells by |mean| for each case):

In [ ]:
for k, g in grids.items():
    st = g.stack().rename("mean").reset_index()
    print(k, st.reindex(st["mean"].abs().sort_values(ascending=False).index).head(5).round(3).to_string(index=False), "\n")
    g.round(4).to_csv(OUT / f"grid_{k}.csv")

## Metric performance per distortion type × level: SRCC within each cell (81 references per cell)

In [ ]:
sr = {}
for m in METRICS:
    rows = [{"dist_name": t, "level": l, "srcc": srcc(aligned(g, m.name), g["mos"])} for (t, l), g in E.groupby(["dist_name", "level"])]
    sr[m.name] = pd.DataFrame(rows).pivot(index="dist_name", columns="level", values="srcc").reindex(grids["A-1"].index)
fig = srcc_grid_figure(sr)
fig.savefig(FIG / "srcc_type_level.png", dpi=130, bbox_inches="tight")
pd.concat(sr, names=["metric"]).round(4).to_csv(REPO_ROOT / "results" / "tables" / "srcc_type_level.csv")
plt.show()

## The five cases: what the compared metrics look at (B-27)

Each figure shows only the views of the metrics that disagree in that case, and every number adds up to the metric's own score:
- **|ΔY| luma / |ΔIQ| chroma** (YIQ, 0–255, shared scale): SSIM and MS-SSIM only see Y, so chroma change is invisible to them.
- **1 − SSIM map**: its mean is 1 − SSIM. **1 − MS-SSIM map**: per-scale maps combined with the MS-SSIM weights (shared scale with SSIM).
- **LPIPS-VGG map**: its mean is the LPIPS score.
- **VGG16 stage bars**: LPIPS per layer, and DISTS per stage split into texture (global feature means) and structure (global correlations). Each metric's bars sum to its score. DISTS compares whole-image statistics, so it has no spatial map; the bars are its honest equivalent.

| case | compares | panels |
|---|---|---|
| A-1 | SSIM vs MS-SSIM | luma, 1−SSIM, 1−MS-SSIM (+ scale figure below) |
| A-2 | LPIPS vs DISTS | LPIPS map, stage bars |
| B-1 | SSIM vs MOS | luma, chroma, 1−SSIM |
| B-2 | DISTS vs MOS | RGB diff (where the damage is), LPIPS map (a local metric for contrast), stage bars |
| C-1 | SSIM vs LPIPS, two images | luma, chroma, 1−SSIM, LPIPS map per image |

In [ ]:
PANELS = {"A-1": ["luma", "ssim", "msssim"], "A-2": ["lpips", "stages"], "B-1": ["luma", "chroma", "ssim"],
          "B-2": ["rgb_diff", "lpips", "stages"]}
for k, panels in PANELS.items():
    p = picks[k]
    fig = explain_figure(E[E["id"] == p["id"]].iloc[0], panels, title=k, ranks=rg.loc[p["id"]])
    fig.savefig(FIG / f"{k}_images.png", dpi=110, bbox_inches="tight")
    plt.show()
p = picks["C-1"]
fig = explain_pair_figure(E[E["id"] == p["id_i"]].iloc[0], E[E["id"] == p["id_j"]].iloc[0], ["luma", "chroma", "ssim", "lpips"],
                          note=f"C-1: SSIM prefers i (Δr={p['d_trad']:+.2f}), LPIPS-VGG prefers j (Δr={p['d_learned']:+.2f}); humans side with {p['human_sides_with']}")
fig.savefig(FIG / "C-1_images.png", dpi=110, bbox_inches="tight")
plt.show()

Numbers behind the stage bars (A-2, B-2):

In [ ]:
from humper.data import load_image
from humper.explain import dists_stages, lpips_layers, STAGES
for k in ["A-2", "B-2"]:
    r = E[E["id"] == picks[k]["id"]].iloc[0]
    ref, dist = load_image(r["ref_path"]), load_image(r["dist_path"])
    tex, st = dists_stages(ref, dist)
    display(pd.DataFrame({"lpips": [float("nan"), *lpips_layers(ref, dist)], "dists_texture": tex, "dists_structure": st}, index=STAGES).round(4).rename_axis(k))

## MS-SSIM scale by scale (single-image cases)

Row 1: the original (reference) image at each MS-SSIM scale (2×2 average pooling between scales; SSIM only uses scale 1).
Row 2: where MS-SSIM sees damage at that scale (1 − cs; 1 − l·cs at scale 5). MS-SSIM = product over scales of (mean)^weight, so the heavily weighted scales 2–4 dominate.

In [ ]:
for k in ["A-1", "A-2", "B-1", "B-2"]:
    p = picks[k]
    fig = msssim_scales_figure(E[E["id"] == p["id"]].iloc[0], title=k)
    fig.savefig(FIG / f"{k}_msssim_scales.png", dpi=110, bbox_inches="tight")
    plt.show()

## How typical is each case? (systematic vs isolated, threshold 0.5 as in D-24)

In [ ]:
sysm = F.systematic(s)
for k in ["B-1", "B-2"]:
    p = picks[k]
    print(k, p["metric"], p["dist_name"], "share of images with |r_m - r_mos| > 0.5:", sysm.loc[p["dist_name"], p["metric"]])